# Expected silage and grain shares by state and year

**Why:** our only field-level labels are the BARC fields in Maryland. For every other state we check the model's
*total* silage area against USDA NASS acreage. This notebook turns the raw NASS download (`corn_acres.csv`) into one
clean reference table, `02_QuickStats/state_silage_ratios.csv`, with one row per state and year.

**How to use the table:** sum the area of the fields the model calls silage and grain in a state and year, compute
`pred_silage_share = silage_area / (silage_area + grain_area)`, and compare it with `silage_share_harvested`.
This checks bias in the total, not field accuracy: errors that cancel out look fine here.

**Training versus validation:** `prior_silage_share_5yr` uses only the five years *before* each row, so it can be a
model input or prior without seeing that year's answer. Do not use the same year's `silage_share_harvested` both to
tune the model and to report validation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)

# Works from the repository root or from 02_QuickStats/ (local machine or Keeboh).
ROOT = Path.cwd().parent if Path.cwd().name == "02_QuickStats" else Path.cwd()
RAW = ROOT / "corn_acres.csv"
OUT = ROOT / "02_QuickStats" / "state_silage_ratios.csv"

LAST_YEAR = 2025          # 2026 silage acres are not published yet
PRIOR_YEARS = 5
ITEMS = {"CORN - ACRES PLANTED": "planted",
         "CORN, GRAIN - ACRES HARVESTED": "grain",
         "CORN, SILAGE - ACRES HARVESTED": "silage"}

## 1. Load and look at what is in the raw file

The download mixes several things that must not be added together:
- **Program:** `SURVEY` (annual estimates) and `CENSUS` (every 5 years, 1997-2022).
- **Period:** the final `YEAR` value plus earlier forecasts (`YEAR - AUG FORECAST`, `YEAR - JUN ACREAGE`, ...).
- **Domain:** `TOTAL` plus Census breakdowns by farm size, sales and industry.
- **Value:** numbers with commas, and `(D)` where NASS withheld the value for privacy.

In [2]:
raw = pd.read_csv(RAW, dtype=str)
raw["year"] = raw["Year"].astype(int)
raw["acres"] = pd.to_numeric(raw["Value"].str.replace(",", "").str.strip(), errors="coerce")
print(f"{len(raw):,} rows, years {raw.year.min()}-{raw.year.max()}, {raw.State.nunique()} state values")
for col in ["Program", "Period", "Geo Level", "Domain", "Data Item"]:
    print(f"\n{col}:", raw[col].value_counts().to_dict())
print("\nwithheld (D) values:", (raw["Value"].str.strip() == "(D)").sum())

39,676 rows, years 1866-2026, 51 state values

Program: {'SURVEY': 21929, 'CENSUS': 17747}

Period: {'YEAR': 35088, 'YEAR - JUN ACREAGE': 1032, 'YEAR - SEP FORECAST': 849, 'YEAR - MAR ACREAGE': 738, 'YEAR - AUG FORECAST': 687, 'YEAR - OCT FORECAST': 628, 'YEAR - NOV FORECAST': 507, 'YEAR - OCT ACREAGE': 98, 'YEAR - AUG ACREAGE': 49}

Geo Level: {'STATE': 38767, 'NATIONAL': 909}

Domain: {'TOTAL': 22526, 'AREA HARVESTED': 4591, 'AREA OPERATED': 3316, 'NAICS CLASSIFICATION': 3311, 'FARM SALES': 2971, 'ECONOMIC CLASS': 2961}

Data Item: {'CORN, GRAIN - ACRES HARVESTED': 19295, 'CORN, SILAGE - ACRES HARVESTED': 13520, 'CORN - ACRES PLANTED': 6861}

withheld (D) values: 2031


## 2. Keep one value per state, year and item

Rules, applied in order:
1. Survey: `Program == SURVEY`, `Period == YEAR`, `Domain == TOTAL`. This is the only source with planted acres
   and the only one published every year.
2. Drop `OTHER STATES`: it is a grab-bag aggregate whose members change between years (values are mostly 0).
3. Keep the national row and label it `US TOTAL`.
4. Census `TOTAL` rows are kept separately as a cross-check for 1997, 2002, 2007, 2012, 2017 and 2022.

In [3]:
def tidy(df):
    df = df[df["Data Item"].isin(ITEMS) & (df["State"] != "OTHER STATES")].copy()
    df["state"] = np.where(df["Geo Level"] == "NATIONAL", "US TOTAL", df["State"])
    df["item"] = df["Data Item"].map(ITEMS)
    assert not df.duplicated(["state", "year", "item"]).any(), "more than one value per state, year and item"
    return df.pivot(index=["state", "year"], columns="item", values="acres").rename_axis(columns=None)

base = (raw["Domain"] == "TOTAL") & (raw["Period"] == "YEAR")
survey = tidy(raw[base & (raw["Program"] == "SURVEY")])
census = tidy(raw[base & (raw["Program"] == "CENSUS")]).add_prefix("census_")
ansi = raw.drop_duplicates("State").set_index("State")["State ANSI"]

print("survey rows:", len(survey), "| census rows:", len(census))
print("withheld or missing in survey:", survey.isna().sum().to_dict())

survey rows: 7604 | census rows: 300
withheld or missing in survey: {'grain': 447, 'planted': 2664, 'silage': 2388}


## 3. Check coverage and gaps

What the checks below found (Nov 14, 2026 download):
- 48 states (no Alaska or Hawaii; NASS does not publish corn there).
- Since 2012 every state has planted and silage acres every year.
- **Grain is never published** for Connecticut, Maine, Massachusetts, Nevada, New Hampshire, Rhode Island and
  Vermont. These are small, silage-dominated states (Vermont: about 90% of corn is silage).
- No 2026 silage acres yet, so the table stops at 2025.
- No row has silage + grain greater than planted.

In [4]:
recent = survey.reset_index().query("2012 <= year <= @LAST_YEAR and state != 'US TOTAL'")
missing = (recent.set_index(["state", "year"]).isna().groupby("state").sum())
print("states:", recent.state.nunique())
print("\nstate-years missing a value, 2012-2025:")
print(missing[missing.any(axis=1)])

over = survey.dropna().query("grain + silage > planted")
print("\nrows where grain + silage > planted:", len(over))

decade = survey.reset_index().assign(decade=lambda d: d.year // 10 * 10)
print("\nstates with planted and silage, by decade:")
print(decade.dropna(subset=["planted", "silage"]).groupby("decade").state.nunique().to_string())

states: 48

state-years missing a value, 2012-2025:
               grain  planted  silage
state                                
CONNECTICUT       14        0       0
MAINE             14        0       0
MASSACHUSETTS     14        0       0
NEVADA            14        0       0
NEW HAMPSHIRE     14        0       0
RHODE ISLAND      14        0       0
VERMONT           14        0       0

rows where grain + silage > planted: 0

states with planted and silage, by decade:
decade
1910     1
1920    49
1930    49
1940    49
1950    49
1960    49
1970    49
1980    48
1990    48
2000    49
2010    49
2020    49


## 4. Fill grain where it is not published, compute shares

- Where grain is missing but planted and silage exist, `grain = planted - silage` and `grain_source = residual`.
  This assumes zero lost acres, so grain is slightly too high and the silage share slightly too low there.
  Lost acres are left blank for those rows.
- `lost = planted - grain - silage`: abandoned, grazed, greenchop and survey error together. A rough balance,
  not an exact identity.
- `silage_share_harvested = silage / (grain + silage)` is the number to compare with the model, because the model
  only classifies fields that were harvested.
- `*_share_planted` columns are kept for reference.

In [5]:
t = survey.join(census, how="left").reset_index()
t = t[(t.year <= LAST_YEAR) & t.planted.notna() & t.silage.notna()].copy()

t["grain_source"] = np.where(t.grain.notna(), "survey", "residual")
t["grain"] = t.grain.fillna(t.planted - t.silage)
t["lost"] = np.where(t.grain_source == "survey", t.planted - t.grain - t.silage, np.nan)

harvested = t.grain + t.silage
t["silage_share_harvested"] = t.silage / harvested
t["grain_share_harvested"] = t.grain / harvested
t["silage_share_planted"] = t.silage / t.planted
t["grain_share_planted"] = t.grain / t.planted
t["lost_share_planted"] = t.lost / t.planted

t["census_silage_share_harvested"] = t.census_silage / (t.census_silage + t.census_grain)
t["survey_vs_census_silage"] = t.silage / t.census_silage

## 5. A prior that does not peek at the current year

`prior_silage_share_5yr` is the mean of `silage_share_harvested` over the previous five years (at least three
needed); `prior_silage_share_5yr_sd` is its spread. Use these as a model input or a "how surprising is this
prediction?" band without leaking that year's answer.

In [6]:
t = t.sort_values(["state", "year"])
past = t.groupby("state")["silage_share_harvested"].shift(1)
roll = past.groupby(t.state).rolling(PRIOR_YEARS, min_periods=3)
t["prior_silage_share_5yr"] = roll.mean().reset_index(level=0, drop=True)
t["prior_silage_share_5yr_sd"] = roll.std().reset_index(level=0, drop=True)
t["prior_n_years"] = past.groupby(t.state).rolling(PRIOR_YEARS, min_periods=1).count().reset_index(level=0, drop=True)

## 6. Write the reference CSV

In [7]:
t["state_ansi"] = t.state.map(ansi).where(t.state != "US TOTAL", "00")
cols = ["state", "state_ansi", "year",
        "planted", "grain", "silage", "lost", "grain_source",
        "silage_share_harvested", "grain_share_harvested",
        "silage_share_planted", "grain_share_planted", "lost_share_planted",
        "prior_silage_share_5yr", "prior_silage_share_5yr_sd", "prior_n_years",
        "census_grain", "census_silage", "census_silage_share_harvested", "survey_vs_census_silage"]
out = t[cols].rename(columns={c: f"{c}_acres" for c in ["planted", "grain", "silage", "lost",
                                                         "census_grain", "census_silage"]})
share_cols = [c for c in out.columns if "share" in c or c == "survey_vs_census_silage"]
out[share_cols] = out[share_cols].round(4)
out.to_csv(OUT, index=False)
print(f"wrote {OUT.relative_to(ROOT)}: {len(out):,} rows, {out.state.nunique()} states incl. US TOTAL, "
      f"years {out.year.min()}-{out.year.max()}")
out[out.year >= 2012].head()

wrote 02_QuickStats/state_silage_ratios.csv: 4,890 rows, 49 states incl. US TOTAL, years 1919-2025


,state,state_ansi,year,planted_acres,grain_acres,silage_acres,lost_acres,grain_source,silage_share_harvested,grain_share_harvested,silage_share_planted,grain_share_planted,lost_share_planted,prior_silage_share_5yr,prior_silage_share_5yr_sd,prior_n_years,census_grain_acres,census_silage_acres,census_silage_share_harvested,survey_vs_census_silage
146,ALABAMA,01,2012,310000.0,295000.0,7000.0,8000.0,survey,0.0232,0.9768,0.0226,0.9516,0.0258,0.0329,0.0079,5.0,285328.0,8371.0,0.0285,0.8362
147,ALABAMA,01,2013,320000.0,295000.0,9000.0,16000.0,survey,0.0296,0.9704,0.0281,0.9219,0.0500,0.0306,0.0089,5.0,NaN,NaN,NaN,NaN
148,ALABAMA,01,2014,300000.0,285000.0,9000.0,6000.0,survey,0.0306,0.9694,0.0300,0.9500,0.0200,0.0284,0.0068,5.0,NaN,NaN,NaN,NaN
149,ALABAMA,01,2015,260000.0,245000.0,9000.0,6000.0,survey,0.0354,0.9646,0.0346,0.9423,0.0231,0.0276,0.0061,5.0,NaN,NaN,NaN,NaN
150,ALABAMA,01,2016,330000.0,315000.0,7000.0,8000.0,survey,0.0217,0.9783,0.0212,0.9545,0.0242,0.0277,0.0063,5.0,NaN,NaN,NaN,NaN


## 7. Sanity checks

**Maryland** (where BARC is): about 8-10% of harvested corn is silage. In Census years the survey and Census can
disagree a lot (2022: survey 40,000 silage acres, Census 22,771), which is real uncertainty in the "truth" we
validate against.

In [8]:
out.query("state == 'MARYLAND' and year >= 2012")[
    ["year", "planted_acres", "grain_acres", "silage_acres", "lost_acres", "silage_share_harvested",
     "prior_silage_share_5yr", "census_silage_acres", "census_silage_share_harvested"]]

,year,planted_acres,grain_acres,silage_acres,lost_acres,silage_share_harvested,prior_silage_share_5yr,census_silage_acres,census_silage_share_harvested
2828,2012,495000.0,435000.0,50000.0,10000.0,0.1031,0.1154,42816.0,0.0895
2829,2013,480000.0,420000.0,55000.0,5000.0,0.1158,0.1115,NaN,NaN
2830,2014,500000.0,430000.0,60000.0,10000.0,0.1224,0.1105,NaN,NaN
2831,2015,440000.0,380000.0,45000.0,15000.0,0.1059,0.1178,NaN,NaN
2832,2016,460000.0,400000.0,50000.0,10000.0,0.1111,0.1144,NaN,NaN
2833,2017,480000.0,420000.0,50000.0,10000.0,0.1064,0.1117,33382.0,0.0706
2834,2018,440000.0,380000.0,45000.0,15000.0,0.1059,0.1123,NaN,NaN
2835,2019,510000.0,460000.0,40000.0,10000.0,0.0800,0.1103,NaN,NaN
2836,2020,475000.0,420000.0,40000.0,15000.0,0.0870,0.1019,NaN,NaN
2837,2021,460000.0,415000.0,35000.0,10000.0,0.0778,0.0981,NaN,NaN


In [9]:
latest = out.query("year == @LAST_YEAR and state != 'US TOTAL'").sort_values("silage_share_harvested", ascending=False)
print(f"{LAST_YEAR}: silage share of harvested corn, highest and lowest")
show = ["state", "silage_share_harvested", "silage_acres", "grain_acres", "grain_source"]
print(latest[show].head(10).to_string(index=False))
print("...")
print(latest[show].tail(5).to_string(index=False))
print("\nUS TOTAL:", out.query("state == 'US TOTAL' and year == @LAST_YEAR").silage_share_harvested.item())

2025: silage share of harvested corn, highest and lowest
        state  silage_share_harvested  silage_acres  grain_acres grain_source
       NEVADA                  0.9231       24000.0       2000.0     residual
      VERMONT                  0.8842       84000.0      11000.0     residual
        MAINE                  0.8333       25000.0       5000.0     residual
NEW HAMPSHIRE                  0.8333       10000.0       2000.0     residual
   CALIFORNIA                  0.8315      370000.0      75000.0       survey
  CONNECTICUT                  0.8261       19000.0       4000.0     residual
      ARIZONA                  0.7937       50000.0      13000.0       survey
MASSACHUSETTS                  0.7857       11000.0       3000.0     residual
        IDAHO                  0.6588      280000.0     145000.0       survey
         UTAH                  0.5976       49000.0      33000.0       survey
...
      state  silage_share_harvested  silage_acres  grain_acres grain_source
   DE

In [10]:
c = out.dropna(subset=["census_silage_share_harvested"]).query("state != 'US TOTAL'")
gap = (c.silage_share_harvested - c.census_silage_share_harvested)
print("survey minus Census silage share, by Census year (median and 90th percentile of |gap|):")
print(c.assign(gap=gap.abs()).groupby("year").gap.describe(percentiles=[0.5, 0.9])[["count", "50%", "90%", "max"]].round(3))

survey minus Census silage share, by Census year (median and 90th percentile of |gap|):
      count    50%    90%    max
year                            
1997   46.0  0.004  0.015  0.035
2002   47.0  0.005  0.034  0.053
2007   48.0  0.002  0.020  0.080
2012   47.0  0.006  0.028  0.125
2017   47.0  0.016  0.083  0.291
2022   48.0  0.021  0.107  0.349


## 8. Comparing model output with the table

Give it one row per state and year with the predicted silage and grain **area in acres** (sum of field areas,
not field counts, because silage and grain fields differ in size).

In [11]:
def compare(pred: pd.DataFrame, ref: pd.DataFrame = out) -> pd.DataFrame:
    """pred columns: state, year, pred_silage_acres, pred_grain_acres."""
    m = pred.merge(ref[["state", "year", "silage_share_harvested", "prior_silage_share_5yr_sd",
                        "census_silage_share_harvested"]], on=["state", "year"], how="left")
    m["pred_silage_share"] = m.pred_silage_acres / (m.pred_silage_acres + m.pred_grain_acres)
    m["share_error"] = m.pred_silage_share - m.silage_share_harvested
    return m

compare(pd.DataFrame({"state": ["MARYLAND"], "year": [2023],
                      "pred_silage_acres": [52_000], "pred_grain_acres": [400_000]}))

,state,year,pred_silage_acres,pred_grain_acres,silage_share_harvested,prior_silage_share_5yr_sd,census_silage_share_harvested,pred_silage_share,share_error
0,MARYLAND,2023,52000,400000,0.0638,0.0124,NaN,0.115044,0.051244
